# BigNeuron 전체 비교 — Ours vs Optimization vs Non-optimization

우리 방법을 BigNeuron의 optimization 결과 및 non-optimization 결과와 함께 비교합니다.

| 그룹 | 출처 | 특징 |
|------|------|------|
| **Ours** | 현재 파이프라인 | Riemannian FMM + refinement |
| **Optimization** | `results/Optimization/` | BigNeuron 최적화 결과 |
| **Non-optimization** | `gold166_no_optimization/` | BigNeuron 기본(비최적화) 결과 |

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

ROOT     = Path('..').resolve()
GOLD_DIR = ROOT / 'data' / 'gold_standard'
OPT_DIR  = ROOT / 'results' / 'Optimization'
NOPT_DIR = ROOT / 'gold166_no_optimization'
OURS_DIR = ROOT / 'results' / 'ours'

# 4개 noisy 샘플 매핑: sample_stem → (noopt 폴더, gold_stem, vxy, vz)
SAMPLE_CFG = {
    'neuron2': {
        'label':     'neuron2',
        'gold_stem': 'neuron2',
        'noopt_dir': NOPT_DIR/'checked6_mouse_culturedcell_Cambridge_in_vivo_2_photon_PAGFP'/'done_neuron2',
        'opt_dir':   OPT_DIR/'neuron2',
        'ours_swc':  OURS_DIR/'neuron2.swc',
        'vxy': 0.2688637, 'vz': 1.0,
    },
    'neuron4': {
        'label':     'neuron4',
        'gold_stem': 'neuron4',
        'noopt_dir': NOPT_DIR/'checked6_mouse_culturedcell_Cambridge_in_vivo_2_photon_PAGFP'/'done_neuron4',
        'opt_dir':   OPT_DIR/'neuron4',
        'ours_swc':  OURS_DIR/'neuron4.swc',
        'vxy': 0.5377274, 'vz': 1.0,
    },
    's06b': {
        'label':     's06b',
        'gold_stem': '1201_01_s06b_L36_Sum_ch2.tif',
        'noopt_dir': NOPT_DIR/'checked6_mouse_korea'/'1201_01_s06b_L36_Sum_ch2',
        'opt_dir':   OPT_DIR/'1201_01_s06b_L36_Sum_ch2',
        'ours_swc':  OURS_DIR/'1201_01_s06b_L36_Sum_ch2.tif.swc',
        'vxy': 0.104, 'vz': 0.5,
    },
    's10mm': {
        'label':     's10mm',
        'gold_stem': '1201_01_s10mm_ch2.tif',
        'noopt_dir': NOPT_DIR/'checked6_mouse_korea'/'1201_01_s10mm_ch2',
        'opt_dir':   OPT_DIR/'1201_01_s10mm_ch2',
        'ours_swc':  OURS_DIR/'1201_01_s10mm_ch2.tif.swc',
        'vxy': 0.104, 'vz': 0.5,
    },
}

THRESHOLDS = [2.0, 5.0]

In [ ]:
def load_pts(path, sxy=1.0, sz=1.0):
    pts = []
    for line in Path(path).read_text().splitlines():
        if line.startswith('#') or not line.strip(): continue
        p = line.split()
        if len(p) < 7: continue
        pts.append([float(p[2])*sxy, float(p[3])*sxy, float(p[4])*sz])
    return np.array(pts, dtype=np.float32) if pts else np.empty((0,3))

def compute_metrics(auto, gold, thr=2.0):
    if len(auto) == 0 or len(gold) == 0:
        return dict(f1=0, precision=0, recall=0, esa=999)
    d_a2g, _ = cKDTree(gold).query(auto)
    d_g2a, _ = cKDTree(auto).query(gold)
    p = float((d_a2g <= thr).mean())
    r = float((d_g2a <= thr).mean())
    return dict(
        f1=2*p*r/(p+r+1e-8),
        precision=p, recall=r,
        esa=float((d_a2g.mean()+d_g2a.mean())/2)
    )

def parse_algo(swc_path):
    """파일명에서 알고리즘 이름 추출"""
    m = re.search(r'\.v3dpbd_(.+)$', Path(swc_path).stem)
    if m: return m.group(1)
    m = re.search(r'\.v3draw_(.+)$', Path(swc_path).stem)
    if m: return m.group(1)
    return Path(swc_path).stem

In [ ]:
rows = []

for stem, cfg in SAMPLE_CFG.items():
    label  = cfg['label']
    vxy, vz = cfg['vxy'], cfg['vz']
    g_path = GOLD_DIR / f"{cfg['gold_stem']}.swc"
    if not g_path.exists():
        print(f'[SKIP] gold 없음: {g_path}'); continue
    gold = load_pts(g_path, vxy, vz)
    print(f'\n=== {label} (gold={len(gold)} nodes) ===')

    def add_row(algo, group, swc_path, s_xy=vxy, s_z=vz, is_um=False):
        try:
            auto = load_pts(swc_path) if is_um else load_pts(swc_path, s_xy, s_z)
            for thr in THRESHOLDS:
                m = compute_metrics(auto, gold, thr)
                rows.append(dict(sample=label, algorithm=algo, group=group,
                                 threshold=thr, n_auto=len(auto), **m))
            print(f'  {group:14s} {algo[:40]:40s}  F1@2={compute_metrics(auto,gold,2.0)["f1"]:.3f}')
        except Exception as e:
            print(f'  ERROR {algo}: {e}')

    # Ours
    if cfg['ours_swc'].exists():
        add_row('ours', 'Ours', cfg['ours_swc'], is_um=True)

    # Non-optimization
    if cfg['noopt_dir'].exists():
        for swc in sorted(cfg['noopt_dir'].glob('*.swc')):
            add_row(parse_algo(swc), 'Non-opt', swc)

    # Optimization
    if cfg['opt_dir'].exists():
        for swc in sorted(cfg['opt_dir'].glob('*.swc')):
            add_row(parse_algo(swc), 'Opt', swc)

df = pd.DataFrame(rows)
print(f'\n총 {len(df)} rows  샘플={df["sample"].nunique()}  알고리즘={df["algorithm"].nunique()}')

## F1@2µm 순위 테이블 (샘플별)

In [ ]:
df2 = df[df['threshold'] == 2.0].copy()

for label in ['neuron2','neuron4','s06b','s10mm']:
    sub = df2[df2['sample']==label].sort_values('f1', ascending=False).reset_index(drop=True)
    if sub.empty: continue

    ours_idx = sub[sub['algorithm']=='ours'].index
    ours_rank = int(ours_idx[0])+1 if len(ours_idx) else None
    n_total = len(sub)

    def hl(row):
        base = [''] * len(row)
        if row['algorithm'] == 'ours':
            return ['background-color:#ffe0dc; font-weight:bold'] * len(row)
        if row['group'] == 'Opt':
            return ['color:#2060a0'] * len(row)
        return base

    print(f'\n━━━ {label}  (ours: {ours_rank}/{n_total}위) ━━━')
    display(
        sub[['group','algorithm','f1','precision','recall','esa','n_auto']]
          .rename(columns={'f1':'F1@2','precision':'P','recall':'R','n_auto':'nodes'})
          .head(15)
          .style
          .apply(hl, axis=1)
          .background_gradient(cmap='YlGn', subset=['F1@2','P','R'], vmin=0, vmax=0.9)
          .background_gradient(cmap='YlGn_r', subset=['esa'], vmin=0, vmax=20)
          .format({'F1@2':'{:.3f}','P':'{:.3f}','R':'{:.3f}','esa':'{:.2f}'})
          .hide(axis='index')
    )

## 그룹별 평균 F1 비교

In [ ]:
# 그룹별 평균
grp = df2.groupby(['group','algorithm'])['f1'].mean().reset_index()
grp_mean = grp.groupby('group')['f1'].agg(['mean','median','max','count']).round(3)
grp_mean.columns = ['Mean F1@2','Median F1@2','Max F1@2','# algos']
grp_mean = grp_mean.loc[['Ours','Non-opt','Opt']] if 'Ours' in grp_mean.index else grp_mean

display(grp_mean.style
    .background_gradient(cmap='YlGn', subset=['Mean F1@2','Median F1@2','Max F1@2'], vmin=0, vmax=0.8)
    .format('{:.3f}', subset=['Mean F1@2','Median F1@2','Max F1@2'])
    .set_caption('그룹별 평균 F1@2µm (4개 샘플 평균)'))

## Ours 백분위 순위 (Non-opt + Opt 합산)

In [ ]:
print(f'샘플별 Ours 순위 (Non-opt + Opt + Ours 전체)')
print('='*60)
for label in ['neuron2','neuron4','s06b','s10mm']:
    sub = df2[df2['sample']==label].sort_values('f1', ascending=False).reset_index(drop=True)
    if sub.empty or 'ours' not in sub['algorithm'].values: continue
    rank = int(sub[sub['algorithm']=='ours'].index[0]) + 1
    n = len(sub)
    f = sub[sub['algorithm']=='ours']['f1'].values[0]
    pct = (n-rank)/(n-1)*100 if n>1 else 100
    grp_counts = sub['group'].value_counts().to_dict()
    print(f'  {label:8s}: {rank:2d}/{n}위  F1={f:.3f}  상위{pct:.0f}%  '
          f'(Opt:{grp_counts.get("Opt",0)}개, Non-opt:{grp_counts.get("Non-opt",0)}개)')

## F1@2µm vs F1@5µm — Ours의 threshold sensitivity

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
colors = {'Ours':'#e05c4b', 'Opt':'#6baed6', 'Non-opt':'#74c476'}

for ax, label in zip(axes, ['neuron2','neuron4','s06b','s10mm']):
    sub2 = df[df['sample']==label]
    for thr, marker in [(2.0,'o'), (5.0,'s')]:
        st = sub2[sub2['threshold']==thr]
        for grp, col in colors.items():
            vals = st[st['group']==grp]['f1'].values
            if len(vals)==0: continue
            y = np.mean(vals)
            lbl = f'{grp} @{thr:.0f}µm' if label=='neuron2' else None
            ax.bar(f'{grp}\n@{thr:.0f}µm', y, color=col,
                   alpha=0.9 if thr==2.0 else 0.5,
                   edgecolor='white', label=lbl)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis='x', labelsize=7)
    ax.grid(axis='y', alpha=0.3)

axes[0].set_ylabel('F1 (group mean)', fontsize=11)
plt.suptitle('F1@2µm vs F1@5µm — Ours(빨강) / Opt(파랑) / Non-opt(초록)', fontsize=12)
plt.tight_layout()
plt.savefig('compare_all_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## Non-opt 대비 Ours 상세 비교

In [ ]:
# Non-opt만 따로 보기
df_noopt = df2[df2['group'].isin(['Non-opt','Ours'])].copy()
pivot = df_noopt.pivot_table(index='algorithm', columns='sample', values='f1').round(3)
pivot['mean'] = pivot.mean(axis=1)
pivot = pivot.sort_values('mean', ascending=False)

def hl_ours(row):
    return ['background-color:#ffe0dc; font-weight:bold' if row.name=='ours' else '' for _ in row]

display(pivot.style
    .apply(hl_ours, axis=1)
    .background_gradient(cmap='YlGn', vmin=0, vmax=0.8)
    .format('{:.3f}', na_rep='-')
    .set_caption('F1@2µm — Non-optimization 알고리즘 vs Ours (mean 기준 정렬)'))